In [1]:
from pathlib import Path
import re

import xarray as xr
import numpy as np
import pandas as pd

In [2]:
PROJ_DIR = Path.cwd().parent.parent
DATA_DIR = PROJ_DIR / 'data'
PROC_DIR = DATA_DIR / 'processed'

In [5]:
SITE = "SIO"
diag = pd.read_csv(f"{SITE}_pace_diagnostics.csv")

# keep only the rows you consider "matched" (adjust if your status label differs)
diag_ok = diag[diag["status"].eq("Swath Covers Station")].copy()

# parse in-situ time from diagnostics (this is the key)
diag_ok["time_utc"] = pd.to_datetime(diag_ok["time"], utc=True, errors="coerce")  # or "time_utc" if that column exists

# parse PACE time from granule_id (or filename column)
tstr = diag_ok["granule_id"].astype(str).str.extract(r"\.(\d{8}T\d{6})\.")[0]
diag_ok["time_pace"] = pd.to_datetime(tstr, format="%Y%m%dT%H%M%S", utc=True)

diag_ok["dt_min"] = (diag_ok["time_pace"] - diag_ok["time_utc"]).abs().dt.total_seconds() / 60.0
diag_ok["dt_min"].describe(percentiles=[.5,.9,.95,.99])


count    113.000000
mean      60.212389
std       41.863416
min        0.033333
50%       50.783333
90%      120.373333
95%      133.043333
99%      153.942000
max      173.350000
Name: dt_min, dtype: float64

In [4]:
diag

,row_id,time,lat,lon,granule_id,status
0,0,2024-03-11 19:15:00+00:00,32.867,-117.257,"Collection: {'Version': '3.1', 'ShortName': 'P...",Swath Covers Station
1,0,2024-03-11 19:15:00+00:00,32.867,-117.257,NaN,ROW_DONE
2,1,2024-03-18 19:00:00+00:00,32.867,-117.257,"Collection: {'ShortName': 'PACE_OCI_L2_AOP', '...",Swath Covers Station
3,1,2024-03-18 19:00:00+00:00,32.867,-117.257,NaN,ROW_DONE
4,2,2024-03-25 19:34:00+00:00,32.867,-117.257,NONE,No Granules Found
...,...,...,...,...,...,...
213,91,2025-12-15 20:06:00+00:00,32.867,-117.257,"Collection: {'Version': '3.1', 'ShortName': 'P...",Swath Covers Station
214,91,2025-12-15 20:06:00+00:00,32.867,-117.257,"Collection: {'ShortName': 'PACE_OCI_L2_AOP', '...",Swath Covers Station
215,91,2025-12-15 20:06:00+00:00,32.867,-117.257,NaN,ROW_DONE
216,92,2026-01-05 20:18:00+00:00,32.867,-117.257,NONE,No Granules Found


In [7]:
link = diag[diag["status"] == "Swath Covers Station"].copy()

# normalize / keep only what you need
link["time_utc"] = pd.to_datetime(link["time"], utc=True, errors="coerce")  # or time_utc if present
link["row_id"] = link["row_id"].astype(int)

# ensure we have the granule filename column you used in extraction
# (adjust col name if needed)
link["granule_id"] = link["granule_id"].astype(str)

# make obs_key to match the zarr coord
link["obs_key"] = link["granule_id"] + "||" + link["row_id"].astype(str)


In [12]:
link

,row_id,time,lat,lon,granule_id,status,time_utc,obs_key
0,0,2024-03-11 19:15:00+00:00,32.867,-117.257,"Collection: {'Version': '3.1', 'ShortName': 'P...",Swath Covers Station,2024-03-11 19:15:00+00:00,"Collection: {'Version': '3.1', 'ShortName': 'P..."
2,1,2024-03-18 19:00:00+00:00,32.867,-117.257,"Collection: {'ShortName': 'PACE_OCI_L2_AOP', '...",Swath Covers Station,2024-03-18 19:00:00+00:00,"Collection: {'ShortName': 'PACE_OCI_L2_AOP', '..."
6,3,2024-04-01 19:27:00+00:00,32.867,-117.257,"Collection: {'Version': '3.1', 'ShortName': 'P...",Swath Covers Station,2024-04-01 19:27:00+00:00,"Collection: {'Version': '3.1', 'ShortName': 'P..."
8,4,2024-04-08 19:53:00+00:00,32.867,-117.257,"Collection: {'Version': '3.1', 'ShortName': 'P...",Swath Covers Station,2024-04-08 19:53:00+00:00,"Collection: {'Version': '3.1', 'ShortName': 'P..."
10,5,2024-04-15 18:58:00+00:00,32.867,-117.257,"Collection: {'ShortName': 'PACE_OCI_L2_AOP', '...",Swath Covers Station,2024-04-15 18:58:00+00:00,"Collection: {'ShortName': 'PACE_OCI_L2_AOP', '..."
...,...,...,...,...,...,...,...,...
206,88,2025-11-17 20:01:00+00:00,32.867,-117.257,"Collection: {'Version': '3.1', 'ShortName': 'P...",Swath Covers Station,2025-11-17 20:01:00+00:00,"Collection: {'Version': '3.1', 'ShortName': 'P..."
209,89,2025-12-01 20:49:00+00:00,32.867,-117.257,"Collection: {'ShortName': 'PACE_OCI_L2_AOP', '...",Swath Covers Station,2025-12-01 20:49:00+00:00,"Collection: {'ShortName': 'PACE_OCI_L2_AOP', '..."
211,90,2025-12-08 20:17:00+00:00,32.867,-117.257,"Collection: {'ShortName': 'PACE_OCI_L2_AOP', '...",Swath Covers Station,2025-12-08 20:17:00+00:00,"Collection: {'ShortName': 'PACE_OCI_L2_AOP', '..."
213,91,2025-12-15 20:06:00+00:00,32.867,-117.257,"Collection: {'Version': '3.1', 'ShortName': 'P...",Swath Covers Station,2025-12-15 20:06:00+00:00,"Collection: {'Version': '3.1', 'ShortName': 'P..."


In [16]:
link.obs_key[0]

"Collection: {'Version': '3.1', 'ShortName': 'PACE_OCI_L2_AOP'}\nSpatial coverage: {'HorizontalSpatialDomain': {'Geometry': {'GPolygons': [{'Boundary': {'Points': [{'Latitude': 40.45937, 'Longitude': -96.95039}, {'Latitude': 34.83776, 'Longitude': -126.84326}, {'Latitude': 17.42484, 'Longitude': -120.11243}, {'Latitude': 22.60362, 'Longitude': -94.98705}, {'Latitude': 40.45937, 'Longitude': -96.95039}]}}]}}}\nTemporal coverage: {'RangeDateTime': {'BeginningDateTime': '2024-03-11T19:58:04Z', 'EndingDateTime': '2024-03-11T20:03:04Z'}}\nSize(MB): 84.57133769989014\nData: ['https://obdaac-tea.earthdatacloud.nasa.gov/ob-cumulus-prod-public/PACE_OCI.20240311T195804.L2.OC_AOP.V3_1.nc']||0"

In [9]:
ds = xr.open_zarr("SIO_rrs_matchups_5x5.zarr")

pace_meta = pd.DataFrame({
    "obs_key": ds["obs_key"].astype(str).values,
    "granule_id": ds["granule_id"].astype(str).values,
    "row_id": ds["row_id"].values.astype(int),
    # NOTE: ds["time"] is not trustworthy right now, so do NOT use it here
    "lat_pace": ds["lat"].values,
    "lon_pace": ds["lon"].values,
})

/tmp/ipykernel_4657/1981699446.py:1: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_unit' or specify 'use_cftime=True'.
  ds = xr.open_zarr("SIO_rrs_matchups_5x5.zarr")


In [10]:
matchups = link.merge(pace_meta, on="obs_key", how="inner", validate="one_to_one")
print("matchups:", len(matchups))

matchups: 0


In [17]:
blob = diag["granule_id"].astype(str)

# Extract the actual granule filename (PACE_OCI....nc)
diag["pace_filename"] = blob.str.extract(r"(PACE_OCI\.\d{8}T\d{6}\.L2\.[A-Za-z0-9_]+\.[A-Za-z0-9_]+\.nc)")[0]

# sanity
print(diag["pace_filename"].head(5))
print("missing filenames:", diag["pace_filename"].isna().sum(), "of", len(diag))

0    PACE_OCI.20240311T195804.L2.OC_AOP.V3_1.nc
1                                           NaN
2    PACE_OCI.20240318T204706.L2.OC_AOP.V3_1.nc
3                                           NaN
4                                           NaN
Name: pace_filename, dtype: object
missing filenames: 101 of 218


In [18]:
diag["row_id"] = diag["row_id"].astype(int)
diag["obs_key"] = diag["pace_filename"] + "||" + diag["row_id"].astype(str)


In [ ]:
link = diag[diag['

In [3]:
# 1) Load the PACE spectra for SIO
SITE = 'SIO'
ds = xr.open_zarr(f"{SITE}_rrs_matchups_5x5.zarr")

# Build a PACE metadata table (1 row per PACE obs):
pace_meta = pd.DataFrame({
    "row_id": ds["row_id"].values,
    "time_pace": pd.to_datetime(ds["time"].values, utc=True, errors="coerce"),
    "lat_pace": ds["lat"].values.astype("float32"),
    "lon_pace": ds["lon"].values.astype("float32"),
    "granule_id": ds["granule_id"].astype(str).values,
    "obs_key": ds["obs_key"].astype(str).values,
})

# Spectra arrays (still lazy until .compute())
wavelength = ds["wavelength"].values.astype("float32")
Rrs_mean = ds["Rrs_mean"].values  # (obs, wavelength)
Rrs_std  = ds["Rrs_std"].values

/tmp/ipykernel_1443/3679439594.py:2: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_unit' or specify 'use_cftime=True'.
  ds = xr.open_zarr(f"{SITE}_rrs_matchups_5x5.zarr")
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/array/core.py:141: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 'time_unit' or specify 'use_cftime=True'.
  c = np.asarray(c)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/array/core.py:141: SerializationWarning: Unable to decode time axis into full numpy.datetime64[ns] objects, continuing using cftime.datetime objects instead, reason: dates out of range. To silence this warning use a coarser resolution 't

In [4]:
stations_df = pd.read_csv(PROC_DIR / "df_calhab.csv")

# sanity check
stations_df.head()
stations_df.columns


Index(['temp (degree_c)', 'chl1 (mg/m3)', 'chl2 (mg/m3)', 'avg_chloro (mg/m3)',
       'sample_id', 'site_code', 'site_name', 'lat', 'lon', 'time_utc',
       'akashiwo_sanguinea (cells/l)', 'alexandrium_spp (cells/l)',
       'dinophysis_spp (cells/l)', 'lingulodinium_polyedra (cells/l)',
       'prorocentrum_spp (cells/l)',
       'pseudo_nitzschia_delicatissima_group (cells/l)',
       'pseudo_nitzschia_seriata_group (cells/l)', 'ceratium_spp (cells/l)',
       'cochlodinium_spp (cells/l)', 'gymnodinium_spp (cells/l)',
       'other_diatoms (cells/l)', 'other_dinoflagellates (cells/l)',
       'total_phytoplankton (cells/l)'],
      dtype='object')

In [6]:
# 2) Load in-situ (CALHAB) and recreate the same row_id for SIO
insitu = (
    stations_df[stations_df.site_code == SITE]
    .copy()
    .reset_index(drop=True)
)
insitu["row_id"] = insitu.index.astype(int)


In [9]:
# 3) Merge: PACE obs ↔ in-situ row
# a) force-parse insitu time safely (strings -> datetime64[ns, UTC])
matchups["time_utc"] = pd.to_datetime(matchups["time_utc"], utc=True, errors="coerce")

# b) parse PACE time from whatever object/cftime/string into pandas, safely
matchups["time_pace"] = pd.to_datetime(matchups["time_pace"], utc=True, errors="coerce")

# c) compute dt in minutes via epoch seconds (int64), with NaT handled
t1 = matchups["time_pace"].view("int64") // 10**9   # seconds since epoch; NaT -> very negative
t0 = matchups["time_utc"].view("int64")  // 10**9

mask = matchups["time_pace"].notna() & matchups["time_utc"].notna()
dt_min = np.full(len(matchups), np.nan, dtype="float64")
dt_min[mask.values] = np.abs(t1[mask].to_numpy() - t0[mask].to_numpy()) / 60.0

matchups["dt_min"] = dt_min


/tmp/ipykernel_1443/3058902234.py:9: FutureWarning: Series.view is deprecated and will be removed in a future version. Use ``astype`` as an alternative to change the dtype.
  t1 = matchups["time_pace"].view("int64") // 10**9   # seconds since epoch; NaT -> very negative
/tmp/ipykernel_1443/3058902234.py:10: FutureWarning: Series.view is deprecated and will be removed in a future version. Use ``astype`` as an alternative to change the dtype.
  t0 = matchups["time_utc"].view("int64")  // 10**9


In [10]:
print("NaT time_pace:", matchups["time_pace"].isna().sum())
print("NaT time_utc :", matchups["time_utc"].isna().sum())

# show extreme dt values if any
matchups.loc[matchups["dt_min"].notna(), ["time_pace","time_utc","dt_min"]].sort_values("dt_min").tail(10)


NaT time_pace: 0
NaT time_utc : 0


,time_pace,time_utc,dt_min
32,1788-08-31 20:34:18.871345152+00:00,2025-06-23 19:22:00+00:00,1.245498e+08
5,1771-01-04 19:43:26.290448384+00:00,2024-08-26 19:05:00+00:00,1.334030e+08
19,1770-03-16 20:08:52.580896768+00:00,2025-01-21 20:35:00+00:00,1.340395e+08
31,1761-02-13 20:34:18.871345152+00:00,2025-06-16 19:41:00+00:00,1.390276e+08
47,1756-02-27 20:59:45.161793536+00:00,2025-11-10 19:52:00+00:00,1.418500e+08
4,1743-06-17 19:43:26.290448384+00:00,2024-08-19 19:22:00+00:00,1.478837e+08
18,1738-08-07 20:08:52.580896768+00:00,2025-01-13 20:11:00+00:00,1.506514e+08
46,1728-08-11 20:59:45.161793536+00:00,2025-11-03 20:11:00+00:00,1.563278e+08
17,1711-01-07 20:08:52.580896768+00:00,2025-01-06 20:17:00+00:00,1.651478e+08
45,1700-11-09 20:59:45.161793536+00:00,2025-10-27 19:14:00+00:00,1.709149e+08


In [11]:
# pace_meta already has granule_id like 'PACE_OCI.20240916T193710.L2.OC_AOP.V3_1.nc'
tstr = pace_meta["granule_id"].str.extract(r"\.(\d{8}T\d{6})\.")[0]
pace_meta["time_pace"] = pd.to_datetime(tstr, format="%Y%m%dT%H%M%S", utc=True)


In [13]:
matchups = pace_meta.merge(insitu, on="row_id", how="left", validate="many_to_one")
#matchups["dt_min"] = (matchups["time_pace"] - matchups["time_utc"]).abs().dt.total_seconds() / 60.0
matchups["time_utc"] = pd.to_datetime(matchups["time_utc"], utc=True, errors="coerce")

# b) parse PACE time from whatever object/cftime/string into pandas, safely
matchups["time_pace"] = pd.to_datetime(matchups["time_pace"], utc=True, errors="coerce")

# c) compute dt in minutes via epoch seconds (int64), with NaT handled
t1 = matchups["time_pace"].view("int64") // 10**9   # seconds since epoch; NaT -> very negative
t0 = matchups["time_utc"].view("int64")  // 10**9

mask = matchups["time_pace"].notna() & matchups["time_utc"].notna()
dt_min = np.full(len(matchups), np.nan, dtype="float64")
dt_min[mask.values] = np.abs(t1[mask].to_numpy() - t0[mask].to_numpy()) / 60.0

matchups["dt_min"] = dt_min


/tmp/ipykernel_1443/18497670.py:9: FutureWarning: Series.view is deprecated and will be removed in a future version. Use ``astype`` as an alternative to change the dtype.
  t1 = matchups["time_pace"].view("int64") // 10**9   # seconds since epoch; NaT -> very negative
/tmp/ipykernel_1443/18497670.py:10: FutureWarning: Series.view is deprecated and will be removed in a future version. Use ``astype`` as an alternative to change the dtype.
  t0 = matchups["time_utc"].view("int64")  // 10**9


In [15]:
matchups["dt_min"].describe(percentiles=[.5,.9,.95,.99])

count       50.000000
mean       462.606667
std       1984.789800
min          0.933333
50%         57.233333
90%        129.063333
95%        155.590000
99%      10087.176833
max      10135.450000
Name: dt_min, dtype: float64

In [14]:
matchups.sort_values("dt_min", ascending=False)[["granule_id","time_pace","time_utc","dt_min"]].head(5)

,granule_id,time_pace,time_utc,dt_min
48,PACE_OCI.20251201T205627.L2.OC_AOP.V3_1.nc,2025-12-01 20:56:27+00:00,2025-11-24 20:01:00+00:00,10135.450000
49,PACE_OCI.20251208T200556.L2.OC_AOP.V3_1.nc,2025-12-08 20:05:56+00:00,2025-12-01 20:49:00+00:00,10036.933333
8,PACE_OCI.20241014T205121.L2.OC_AOP.V3_1.nc,2024-10-14 20:51:21+00:00,2024-10-14 17:58:00+00:00,173.350000
37,PACE_OCI.20250902T204553.L2.OC_AOP.V3_1.nc,2025-09-02 20:45:53+00:00,2025-09-02 18:32:00+00:00,133.883333
39,PACE_OCI.20250908T210429.L2.OC_AOP.V3_1.nc,2025-09-08 21:04:29+00:00,2025-09-08 18:52:00+00:00,132.483333


In [ ]:
# 4) Validate alignment (do not skip this)
# A) Did every PACE obs find an in-situ row?
n_missing = matchups["station"].isna().sum()
print("Missing in-situ rows for PACE obs:", n_missing, "out of", len(matchups))

In [ ]:
# B) How many unique in-situ rows are covered?
print("Unique row_id in PACE:", matchups["row_id"].nunique())
print("Unique row_id in in-situ SIO:", insitu["row_id"].nunique())

In [ ]:
# C) Any duplicated obs_keys? (should not happen)
dup = matchups["obs_key"].duplicated().sum()
print("Duplicated obs_key:", dup)


In [ ]:
# 5) Produce IO-ready objects; X and wavelength grid
X = np.asarray(ds["Rrs_mean"].values, dtype="float32")  # (obs, wavelength)
wv = ds["wavelength"].values.astype("float32")


In [ ]:
# y (choose target columns from matchups; organism counts columns)
target_cols = [c for c in matchups.columns if c.endswith("(cells/l)")]  # example heuristic
y = matchups[target_cols].copy()


In [ ]:
# keep a provenance index table
provenance = matchups[["row_id","obs_key","granule_id","time_utc","time_pace","dt_min","lat_pace","lon_pace"]].copy()
